In [2]:
import pandas as pd

# 1. Load data
df = pd.read_csv("online_retail.csv")

# 2. Data cleaning
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df = df.dropna(subset=["CustomerID"])

# 3. Convert date
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# 4. Create Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# 5. Remove duplicates
df = df.drop_duplicates()

# 6. Remove non-product categories
df = df[~df["Description"].isin(["POSTAGE", "Manual"])]

# 7. Create Month for analysis
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

# =========================
# 📊 BASIC ANALYSIS
# =========================

# Top products
top_products = (
    df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Monthly revenue
monthly_revenue = (
    df.groupby("Month")["Revenue"]
    .sum()
    .sort_index()
)

# Top customers
customer_revenue = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

# =========================
# 🧠 RFM
# =========================

today = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (today - x.max()).days,
    "InvoiceNo": "nunique",
    "Revenue": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm["Segment"] = "Other"
rfm.loc[(rfm["Recency"] < 30) & (rfm["Frequency"] > 10), "Segment"] = "Loyal"
rfm.loc[(rfm["Recency"] < 30) & (rfm["Monetary"] > 1000), "Segment"] = "VIP"
rfm.loc[(rfm["Recency"] > 90), "Segment"] = "Lost"

# =========================
# 📈 COHORT
# =========================

df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["CohortMonth"] = df.groupby("CustomerID")["InvoiceMonth"].transform("min")

df["CohortIndex"] = (
    (pd.to_datetime(df["InvoiceMonth"]).dt.year - pd.to_datetime(df["CohortMonth"]).dt.year) * 12 +
    (pd.to_datetime(df["InvoiceMonth"]).dt.month - pd.to_datetime(df["CohortMonth"]).dt.month)
)

cohort = (
    df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"]
    .nunique()
    .reset_index()
)

cohort_pivot = cohort.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="CustomerID"
)

cohort_size = cohort_pivot.iloc[:, 0]

retention = cohort_pivot.divide(cohort_size, axis=0)

# =========================
# 💾 SAVE FILES
# =========================

df.to_csv("cleaned_data_v2.csv", index=False)
retention.to_csv("retention.csv")